# 02 - Collecte des données: Airbus Helicopters supply chain

## Source 1 · [Kaggle] · Aerospace Supply Chain Performance & Forecasting
Je récupère ma source Kaggle (jeu de données synthétique) avec l'API de Kaggle, par un script Python.

Les fichiers sont rangés dans `data/raw/` et ne sont jamais modifiés.

In [6]:
# J'installe kagglehub : l'outil officiel de Kaggle qui permet à Python de télécharger un jeu de données
%pip install kagglehub

# J'importe les outils dont j'ai besoin dans ce notebook
import kagglehub        
import pandas as pd     

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Le point d'accès : l'identifiant du jeu de données sur Kaggle
dataset = "robertocarlost/aerospace-supply-chain-performance-and-forecasting"      

# Les fichiers que je veux récupérer (je laisse de côté quality_incidents.csv, dont je n'ai pas besoin)
fichiers = ["purchase_orders.csv", "parts_master.csv", "supply_chain_history.csv"]  

# Je range chaque fichier dans data/raw/
for nom in fichiers:
    chemin = kagglehub.dataset_download(dataset, path=nom, output_dir="../data/raw")
    print(nom, "-> enregistré dans", chemin)  
    # si le téléchargement s'est bien passé, je vois le chemin du fichier

purchase_orders.csv -> enregistré dans ../data/raw\purchase_orders.csv
parts_master.csv -> enregistré dans ../data/raw\parts_master.csv
supply_chain_history.csv -> enregistré dans ../data/raw\supply_chain_history.csv


In [9]:
# Je relis mes 3 fichiers pour vérifier qu'ils sont complets
purchase_orders = pd.read_csv("../data/raw/purchase_orders.csv")
parts_master = pd.read_csv("../data/raw/parts_master.csv")
supply_chain_history = pd.read_csv("../data/raw/supply_chain_history.csv")

# Je dois obtenir : (29666, 9), (300, 9) et (280800, 11)
print("purchase_orders :", purchase_orders.shape)
print("parts_master :", parts_master.shape)
print("supply_chain_history :", supply_chain_history.shape)

purchase_orders : (29666, 9)
parts_master : (300, 9)
supply_chain_history : (280800, 11)


## Source 2 · [Réserve fédérale de New York] · Indice GSCPI
Le GSCPI mesure chaque mois la pression sur les chaînes d'approvisionnement mondiales (0 = niveau moyen).
Je le récupère par un script Python, qui télécharge le fichier Excel depuis son URL.
Le script est reproductible : je peux le relancer chaque mois pour obtenir les nouvelles valeurs

In [6]:
%pip install requests openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests                 
from datetime import date       

# Le point d'accès : l'URL du fichier Excel du GSCPI (New York Fed)
url = "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xlsx"  

entetes = {"User-Agent": "FormationDataSuits/1.0 (projet de formation)"}

# Je demande le fichier au site
reponse = requests.get(url, headers=entetes)
print(reponse.status_code)                 # 200 = tout va bien
reponse.raise_for_status()                 
print("Collecte faite le", date.today())   # je note la date : utile pour mon README

200
Collecte faite le 2026-11-24


In [10]:
# J'enregistre le fichier tel quel sans le modifier, dans data/raw/
with open("../data/raw/gscpi_data.xls", "wb") as fichier:
    fichier.write(reponse.content)         # reponse.content = le contenu du fichier téléchargé

print("Fichier enregistré :", len(reponse.content), "octets")

Fichier enregistré : 252416 octets


In [11]:
import pandas as pd

# J'ouvre le fichier Excel et je regarde les onglets qu'il contient ainsi que les premières lignes
excel = pd.ExcelFile("../data/raw/gscpi_data.xls")
print(excel.sheet_names)

pd.read_excel(excel, sheet_name=0, header=None).head(15)

['GSCPI Monthly Data']


,0,1
0,Date,GSCPI
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,31-Jan-1998,-1.17909
6,28-Feb-1998,-0.489883
7,31-Mar-1998,-0.071228
8,30-Apr-1998,-0.146184
9,31-May-1998,-0.471055


## Source 3 · [APU Frankfurter] · Taux de change EUR/USD
L'API Frankfurter donne gratuitement, et sans clé, les taux de change publiés par la Banque centrale européenne.
Je récupère le taux EUR/USD jour par jour, de 2022 à 2024 (la période de mes données), par un script Python.
Le script est reproductible : je peux le relancer à tout moment.

In [12]:
import json

# Le point d'accès : l'adresse URL de l'API Frankfurter (version actuelle, v2)
url = "https://api.frankfurter.dev/v2/rates"            

# Les paramètres : ce que je demande à l'API
parametres = {
    "from": "2022-01-01",     # première date 2022-01-01: le début de mes données
    "to": "2024-12-31",       # dernière date 2024-12-31: la fin de mes données
    "base": "EUR",            # je pars de l'euro
    "quotes": "USD",          # et je veux seulement le dollar
}

# J'envoie ma demande à l'API
reponse = requests.get(url, params=parametres)
print(reponse.status_code)                 # 200 = tout va bien
reponse.raise_for_status()                 # si ça a échoué, Python s'arrête ici avec un message clair
print("Collecte faite le", date.today())   # je note la date pour mon README

200
Collecte faite le 2026-11-24


In [13]:
# J'enregistre la réponse de l'API telle que je l'ai reçue, sans la modifier
with open("../data/raw/taux_change_eur_usd.json", "w", encoding="utf-8") as fichier:
    json.dump(reponse.json(), fichier, ensure_ascii=False, indent=2)
print("Fichier enregistré dans data/raw/")

Fichier enregistré dans data/raw/


In [17]:
# Je regarde ce que l'API m'a renvoyé : le type et la réponse sous forme de tableau
donnees = reponse.json()
print(type(donnees))

taux = pd.DataFrame(donnees)
print(taux.shape)       # environ 1096 lignes : tous les jours du 1er janvier 2022 au 31 décembre 2024
taux.head()

<class 'list'>
(1096, 4)


,date,base,quote,rate
0,2022-01-01,EUR,USD,1.1320
1,2022-01-02,EUR,USD,1.1320
2,2022-01-03,EUR,USD,1.1327
3,2022-01-04,EUR,USD,1.1297
4,2022-01-05,EUR,USD,1.1303


## Pour finir

- [X] Source 1 (Kaggle) récupérée **par un script Python**, via l'API de Kaggle (kagglehub), une seule fois. Les 3 fichiers CSV sont dans `data/raw/`.
- [X] Source 2 (GSCPI) récupérée **par un script Python** : fichier Excel téléchargé depuis son URL. Le fichier est dans `data/raw/gscpi_data.xls`. À relancer chaque mois pour avoir les nouvelles valeurs
- [X] Source 3 (taux de change EUR/USD) récupérée **par un script Python, via l'API Frankfurter** (sans clé). La réponse brute est dans `data/raw/taux_change_eur_usd.json`.
- [X] J'ai rouvert chaque fichier pour vérifier son contenu.
- [X] Aucune clé ni aucun mot de passe n'est écrit dans ce notebook.
- [X] Dans ma fiche projet, j'ai noté pour chaque source comment je la récupère (fichier, API, page web) et à quel rythme (une fois, chaque jour…).